# When `NA` means none, and when it means unknown

Every missing cell in these tables is the string `NA`. There are no empty strings. The codebook assigns "none" to some of those strings and leaves the others as unknown.

| Column | Rows with `NA` | Reading |
|---|---:|---|
| `PoolQC` | 1453 | no pool; the other 7 have `PoolArea` above 0 |
| `MiscFeature` | 1406 | no miscellaneous feature |
| `Alley` | 1369 | no alley; the other 91 name Gravel or Paved |
| `Fence` | 1179 | no fence |
| `FireplaceQu` | 690 | no fireplace, and these are exactly the rows with `Fireplaces` 0 |
| `LotFrontage` | 259 | the frontage was not recorded |
| garage type, year, finish, quality, condition | 81 each | no garage, and `GarageCars` is 0 |
| `BsmtQual` and `BsmtCond` and `BsmtFinType1` | 37 each | no basement |
| `BsmtExposure` and `BsmtFinType2` | 38 each | 37 of them are the houses with no basement |
| `MasVnrType` and `MasVnrArea` | 8 each | unknown; "no veneer" is the word `None`, and 864 rows already say that |
| `Electrical` | 1 | unknown |

Two basement rows keep the word "none" from being automatic.

House 949 has basement quality `Gd`, condition `TA`, and 936 square feet of basement. Its exposure cell is `NA`. The basement is there. The exposure is the missing fact.

House 333 has a basement of 3206 square feet, a first finished area of 1124, and a second finished area of 479. `BsmtFinType2` is `NA`. The square feet were recorded. The name of that second finish was not.

The test file adds blanks the training file never has, including 4 missing zoning cells and 1 missing `GarageCars`. A rule that saw a filled column on all 1460 training rows can still meet `NA` on a test row.


In [2]:
# Locate the contest tables beside this notebook, or under the course root.
from pathlib import Path
import csv

def manifest_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "15-House Prices" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "data_description.txt").exists():
            return direct
    raise FileNotFoundError(name)

def read_manifest(name):
    with manifest_path(name).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

train = read_manifest("train.csv")
test = read_manifest("test.csv")
submission = read_manifest("sample_submission.csv")
print(len(train), len(test), len(submission))
print(train[0]["Id"], train[0]["SalePrice"])


1460 1459 1459
1 208500


In [3]:
# Count the token NA, then separate "none" from the two odd basements.
from collections import Counter
counts = Counter()
for row in train:
    for column, value in row.items():
        if value == "":
            raise AssertionError(column)
        if value == "NA":
            counts[column] += 1
expected = {
    "PoolQC": 1453,
    "MiscFeature": 1406,
    "Alley": 1369,
    "Fence": 1179,
    "FireplaceQu": 690,
    "LotFrontage": 259,
    "GarageType": 81,
    "GarageYrBlt": 81,
    "GarageFinish": 81,
    "GarageQual": 81,
    "GarageCond": 81,
    "BsmtExposure": 38,
    "BsmtFinType2": 38,
    "BsmtQual": 37,
    "BsmtCond": 37,
    "BsmtFinType1": 37,
    "MasVnrType": 8,
    "MasVnrArea": 8,
    "Electrical": 1,
}
for column, count in expected.items():
    print(column, counts[column])
assert dict(counts) == expected
assert sum(int(row["PoolArea"]) > 0 for row in train) == 7
assert sum(row["Fireplaces"] == "0" for row in train) == 690
assert all(
    (row["FireplaceQu"] == "NA") == (row["Fireplaces"] == "0") for row in train
)
assert all(
    (row["GarageType"] == "NA") == (row["GarageCars"] == "0") for row in train
)
assert sum(row["MasVnrType"] == "None" for row in train) == 864
by_id = {row["Id"]: row for row in train}
house_949 = by_id["949"]
house_333 = by_id["333"]
print("949", house_949["BsmtQual"], house_949["TotalBsmtSF"], house_949["BsmtExposure"])
print("333", house_333["BsmtFinSF2"], house_333["BsmtFinType2"], house_333["TotalBsmtSF"])
assert house_949["BsmtQual"] == "Gd"
assert int(house_949["TotalBsmtSF"]) == 936
assert house_949["BsmtExposure"] == "NA"
assert int(house_333["BsmtFinSF2"]) == 479
assert house_333["BsmtFinType2"] == "NA"
assert int(house_333["TotalBsmtSF"]) == 3206


PoolQC 1453
MiscFeature 1406
Alley 1369
Fence 1179
FireplaceQu 690
LotFrontage 259
GarageType 81
GarageYrBlt 81
GarageFinish 81
GarageQual 81
GarageCond 81
BsmtExposure 38
BsmtFinType2 38
BsmtQual 37
BsmtCond 37
BsmtFinType1 37
MasVnrType 8
MasVnrArea 8
Electrical 1
949 Gd 936 NA
333 479 NA 3206


## Pitfall

Filling every `NA` with the word `None` is right for an alley and wrong for house 949. That house has a basement. Lot frontage is the other trap: 259 missing frontages are unknown lengths, and the codebook never defines them as zero.

## Summary

`NA` is "none" for the pool, the alley, the fence, the fireplace, and 81 garages. It is unknown for 259 frontages, 8 veneers, and 1 electrical system. Houses 949 and 333 show a basement `NA` that is not "no basement."
